# Lab 4 — SOLUTION

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

df = pd.read_csv("data/fraud.csv")
X, y = df.drop(columns="is_fraud"), df.is_fraud
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

base = KNeighborsClassifier().fit(Xtr, ytr)
pred = base.predict(Xte)
print(f"baseline: acc={accuracy_score(yte,pred):.3f} recall={recall_score(yte,pred):.3f}")

results = []
for name, sampler in [("under", RandomUnderSampler(random_state=42)),
                      ("over", RandomOverSampler(random_state=42)),
                      ("SMOTE", SMOTE(random_state=42))]:
    pipe = Pipeline([("s", sampler), ("m", KNeighborsClassifier())]).fit(Xtr, ytr)
    p = pipe.predict(Xte)
    results.append((name, accuracy_score(yte,p), recall_score(yte,p), f1_score(yte,p)))
print(pd.DataFrame(results, columns=["strategy","accuracy","recall","F1"]).round(3))

**Teaching notes:** the baseline shows ~99% accuracy with near-zero recall — the 'lying metric' reveal. imblearn Pipeline resamples inside CV folds only; resampling before the split is the classic bug (revisited in Lab 16).